# *Global Settings*

In [ ]:
import pandas as pd
import numpy as np
import osmnx as ox
import geopandas as gpd
from pathlib import Path
import pickle
import os
import warnings
import networkx as nx

warnings.filterwarnings('ignore')

# ==============================================================================
# 0. CONFIGURACIÓN DE RUTAS PORTABLES
# ==============================================================================
BASE_DIR = Path.cwd()

# --- DEFINICIÓN DE CARPETAS --- 
INPUTS_DIR = BASE_DIR / "Inputs"
INFRA_DIR = INPUTS_DIR / "Infrastructure"
RATES_DIR = INPUTS_DIR / "Emission rates"
GPS_DIR = INPUTS_DIR / "GPS User Data"

OUTPUTS_DIR = BASE_DIR / "Outputs"
INTERMEDIATE_DIR = OUTPUTS_DIR / "Intermediate Outputs"
FINAL_DIR = OUTPUTS_DIR / "Final Outputs"

# Creation of folders
for folder in [INFRA_DIR, RATES_DIR, GPS_DIR, INTERMEDIATE_DIR, FINAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# --- DEFINICIÓN DE ARCHIVOS ESPECÍFICOS ---
FILE_GRAFO = INFRA_DIR / "monterrey_drive_network_V1.pkl"
FILE_GRAFO_WALK = INFRA_DIR / "monterrey_walk_network_EydanV1.pkl" 

FILE_METRO = INFRA_DIR / "lineas_metrorrey.csv"
FILE_BUS = INFRA_DIR / "rutas_transporte.geojson"
FILE_GPS_ORIGINAL = GPS_DIR / "top_20users_1_month.parquet"
FILE_MOVES = RATES_DIR / "cleaned_emission_rates_formatted_SB.parquet"

# Archivos de Salida
FILE_INTERMEDIO = INTERMEDIATE_DIR / "Rutas_Completadas_Clasificadas.parquet"
FILE_FINAL_PARQUET = FINAL_DIR / "Resultados_Datos_Emisiones_GPS.parquet"
FILE_FINAL_CSV = FINAL_DIR / "Resultados_Mapa_Emisiones_GPS_Kepler.csv"
FILE_FINAL_GEOJSON = FINAL_DIR / "Resultados_Mapa_Emisiones_GPS_Kepler.geojson"

print(f"Carpetas listas en: {BASE_DIR}")

# **MODULE 1: Código de Modos de Transporte**

## M1 Input files and settings

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import KDTree
from shapely import wkt
import shapely.geometry

print("Cargando infraestructura de transporte...")
bus_routes = gpd.read_file(FILE_BUS)
subway_df = pd.read_csv(FILE_METRO)

# Convertimos la columna de texto a geometría espacial
if 'geometry' in subway_df.columns:
    subway_df['geometry'] = subway_df['geometry'].apply(wkt.loads)
    subway_routes = gpd.GeoDataFrame(subway_df, geometry='geometry', crs="EPSG:4326")
elif 'WKT' in subway_df.columns:
    subway_df['geometry'] = subway_df['WKT'].apply(wkt.loads)
    subway_routes = gpd.GeoDataFrame(subway_df, geometry='geometry', crs="EPSG:4326")
elif 'lat' in subway_df.columns and 'lon' in subway_df.columns:
    subway_routes = gpd.GeoDataFrame(subway_df, geometry=gpd.points_from_xy(subway_df.lon, subway_df.lat), crs="EPSG:4326")
else:
    raise ValueError("¡ALERTA! Revisa las columnas del archivo del metro.")

# ---------------------------------------------------------
# CORRECCIÓN METODOLÓGICA: PROYECCIÓN Y CACHÉ ESPACIAL
# ---------------------------------------------------------
bus_routes = bus_routes.to_crs("EPSG:32614")
subway_routes = subway_routes.to_crs("EPSG:32614")

print("Generando Índices Espaciales (R-Trees) globales...")
_ = subway_routes.sindex 
_ = bus_routes.sindex
print("Infraestructura lista.")

## M1 Functions y Asignación Modal

In [ ]:
import networkx as nx

def calcular_cercania_infraestructura(df, subway_routes, bus_routes):
    """Calcula cercanía al metro/bus usando distancias MÉTRICAS exactas a la línea."""
    
    # 1. Proyectar temporalmente a UTM 14N
    gdf_pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")
    gdf_pts = gdf_pts.to_crs("EPSG:32614")
    
    RADIO_BUSQUEDA_METROS = 100 
    
    # 2. Búsqueda Espacial Exacta (sjoin_nearest mide a la LÍNEA, no al centroide)
    # --- METRO ---
    metro_join = gpd.sjoin_nearest(gdf_pts, subway_routes, how='left', distance_col='dist_metro')
    metro_join = metro_join[~metro_join.index.duplicated(keep='first')] # Limpiar empates de distancia
    df['near_subway_line'] = (metro_join['dist_metro'] < RADIO_BUSQUEDA_METROS).astype(int)
    
    # --- BUS ---
    bus_join = gpd.sjoin_nearest(gdf_pts, bus_routes, how='left', distance_col='dist_bus')
    bus_join = bus_join[~bus_join.index.duplicated(keep='first')] 
    df['near_bus_route'] = (bus_join['dist_bus'] < RADIO_BUSQUEDA_METROS).astype(int)
    
    return df

def clasificar_viajes(df):
    """Modelo Bayesiano completo (4 variables) original, 100% Vectorizado y Normalizado."""
    modos = ['Carro', 'Bus', 'Metro', 'Caminar']
    
    # --- MATRICES ORIGINALES (Tomadas de 4_Clasificación.ipynb) ---
    Cercania = np.array([[0.10, 0.10, 0.80, 0.00],
                        [0.10, 0.80, 0.00, 0.10],
                        [0.40, 0.25, 0.05, 0.30]])

    Velocidad = np.array([[0.05, 0.10, 0.15, 0.60],
                        [0.47, 0.38, 0.05, 0.10],
                        [0.50, 0.30, 0.20, 0.00],
                        [1.00, 0.00, 0.00, 0.00]])

    Distancia = np.array([[0.10, 0.20, 0.30, 0.40],
                        [0.25, 0.25, 0.30, 0.20],
                        [0.40, 0.15, 0.25, 0.20],
                        [0.60, 0.30, 0.00, 0.10],
                        [0.40, 0.40, 0.00, 0.20]])

    Velprom = np.array([[0.10, 0.10, 0.20, 0.60],
                        [0.40, 0.25, 0.25, 0.10]])

    # --- PREPARAR VARIABLES FALTANTES POR VIAJE ---
    # Calculamos la distancia total y velocidad promedio del viaje al vuelo para este usuario
    mask_viajes = df['trip'] > 0
    df['dist_total_km'] = 0.0
    df['avg_speed_trip'] = 0.0
    
    if mask_viajes.any():
        df.loc[mask_viajes, 'dist_total_km'] = df[mask_viajes].groupby('trip')['dis lineal [m]'].transform('sum') / 1000.0
        df.loc[mask_viajes, 'avg_speed_trip'] = df[mask_viajes].groupby('trip')['Speed [km/h]'].transform('mean')

    # --- PASO 1: ÍNDICES VECTORIZADOS (Filtros de variables) ---
    # Cercanía: 0 (Metro), 1 (Bus), 2 (Ninguno)
    idx_c = np.where(df['near_subway_line'] == 1, 0, 
            np.where(df['near_bus_route'] == 1, 1, 2))

    idx_v = np.digitize(df['Speed [km/h]'].fillna(0), bins=[6.001, 20.001, 80.001])
    idx_d = np.digitize(df['dist_total_km'], bins=[1.0, 6.001, 10.001, 18.001])
    idx_vp = np.digitize(df['avg_speed_trip'], bins=[6.001])
    
    # --- PASO 2: BAYES Y NORMALIZACIÓN POR PUNTOS ---
    # Multiplicación matricial (No normalizada)
    P_unnorm = Cercania[idx_c] * Velocidad[idx_v] * Distancia[idx_d] * Velprom[idx_vp]
    
    # Normalización: Hacemos que las probabilidades de los 4 modos sumen 1.0 en cada fila
    suma_puntos = P_unnorm.sum(axis=1, keepdims=True)
    suma_puntos[suma_puntos == 0] = 1 # Prevenir división por cero
    P_norm_puntos = P_unnorm / suma_puntos
    
    df_probs = pd.DataFrame(P_norm_puntos, columns=modos, index=df.index)
    df_probs['trip'] = df['trip']
    
    # --- PASO 3: SUMA DE PROBABILIDADES Y VOTACIÓN POR VIAJE ---
    viajes_df = df_probs[df_probs['trip'] > 0]
    
    if not viajes_df.empty:
        # Acumulamos el peso probabilístico de todos los puntos GPS dentro de un mismo viaje
        suma_por_viaje = viajes_df.groupby('trip')[modos].sum()
        # Seleccionamos el modo que obtuvo la mayor probabilidad global para ese viaje
        modos_ganadores = suma_por_viaje.idxmax(axis=1) 
        df['modo_transporte'] = df['trip'].map(modos_ganadores)
    else:
        df['modo_transporte'] = np.nan
        
    # --- PASO 4: LIMPIEZA DE PARADAS Y MEMORIA ---
    df.loc[df['trip'] <= 0, 'modo_transporte'] = 'Parada'
    df.drop(columns=['dist_total_km', 'avg_speed_trip'], inplace=True, errors='ignore')
    
    return df

# **MODULE 2: Completado de Rutas**

### M2 Input files and settings

In [ ]:
import pandas as pd
import numpy as np
import osmnx as ox
import igraph as ig
import json
import geopandas as gpd
from joblib import Parallel, delayed
from datetime import timedelta
import pyproj
from shapely.ops import transform, substring
from shapely import wkt
from shapely.geometry import Point, LineString
from pyproj import Transformer
from shapely.geometry import Point, LineString
import networkx as nx

# ---------------------------------------------------------
# TRANSFORMADORES GLOBALES
# ---------------------------------------------------------
TRANSFORMER_TO_UTM = Transformer.from_crs("EPSG:4326", "EPSG:32614", always_xy=True)
TRANSFORMER_TO_WGS = Transformer.from_crs("EPSG:32614", "EPSG:4326", always_xy=True)

# Guardamos una versión proyectada global del metro para la interpolación
subway_routes_proj = subway_routes.to_crs("EPSG:32614")

### M2 Functions

In [ ]:
import networkx as nx

def get_ids(df):
    return np.array(df['caid'].unique())

def assign_trips(df):
    """
    Segmentación con Filtro Anti-Teletransportación y 
    Acumulador de Tiempo Estacionario (Low-Pass Filter) Corregido.
    """
    df = df.copy()
    df['Speed [km/h]'] = df['Speed [km/h]'].fillna(0.0)
    df['dis lineal [m]'] = df['dis lineal [m]'].fillna(0.0)
    df['travel time'] = df['travel time'].fillna(pd.Timedelta(seconds=0))

    speeds = df['Speed [km/h]'].tolist()
    travel_times = df['travel time'].dt.total_seconds().tolist() 
    distances = df['dis lineal [m]'].tolist()

    trips = []
    trip_counter = 0
    stop_counter = 0
    
    STOP_SPEED = 3.0  
    STOP_TIME = 300   
    T_MAX_TELEPORT = 1800 

    accumulated_stop_time = 0

    for i in range(len(speeds)):
        speed = speeds[i]
        tt = travel_times[i]
        dist = distances[i]
        
        previous_trip = trips[-1] if i > 0 else None

        # 1. Filtro Anti-Teletransportación
        if tt > T_MAX_TELEPORT:
            stop_counter -= 1
            current_trip = stop_counter
            accumulated_stop_time = 0
            
        else:
            # 2. Detección de quietud
            if speed < STOP_SPEED and dist < 100:
                accumulated_stop_time += tt # Sumamos el tiempo de este pequeño ping
                
                if accumulated_stop_time > STOP_TIME:
                    # Ya estuvo quieto más de 5 mins acumulados. Es una Parada real.
                    if previous_trip is None or previous_trip > 0:
                        stop_counter -= 1
                        current_trip = stop_counter
                    else:
                        current_trip = previous_trip
                else:
                    # Está quieto, pero lleva menos de 5 mins. Sigue siendo Viaje.
                    current_trip = previous_trip if previous_trip is not None else 1
                    
            # 3. Detección de movimiento
            else:
                accumulated_stop_time = 0 
                
                if previous_trip is None or previous_trip < 0:
                    trip_counter += 1
                    current_trip = trip_counter
                else:
                    current_trip = previous_trip
                    
        trips.append(current_trip)
        
    df['trip'] = trips
    return df


# Cálculo de distancia geodésica mediante fórmula de Haversine vectorizada
def haversine_vectorized(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    distance = 6371 * c  
    return distance

# Filtrado de identificadores con densidad de datos insuficiente
def delete_ids_with_few_rows(df, id_list, threshold=2):
    id_counts = df[df['caid'].isin(id_list)]['caid'].value_counts()
    valid_ids = id_counts[id_counts > threshold].index
    return df[df['caid'].isin(valid_ids)]

def _crear_curva_metro(p_start, p_end, df_metro_proj, num_points):
    """Interpola un trayecto de metro obligándolo a seguir la línea específica correcta."""
    
    # 1. Proyectar a UTM 14N usando el transformer global
    x_start, y_start = TRANSFORMER_TO_UTM.transform(p_start.x, p_start.y)
    x_end, y_end = TRANSFORMER_TO_UTM.transform(p_end.x, p_end.y)
    pt_start = Point(x_start, y_start)
    pt_end = Point(x_end, y_end)
    
    # 2. Identificar a qué línea específica de metro pertenecen
    distancias_start = df_metro_proj.distance(pt_start)
    linea_mas_cercana_idx = distancias_start.idxmin()
    nearest_line = df_metro_proj.loc[linea_mas_cercana_idx, 'geometry']
    
    # 3. Proyectar distancias SOLO sobre esa línea particular
    d_start = nearest_line.project(pt_start)
    d_end = nearest_line.project(pt_end)
    
    # 4. Extraer el segmento exacto (soportando viajes en ambas direcciones)
    if d_start <= d_end:
        sub_line = substring(nearest_line, d_start, d_end)
    else:
        sub_line = substring(nearest_line, d_end, d_start)
        # Invertir coordenadas para que la lista siga la dirección del viaje real
        sub_line = LineString(list(sub_line.coords)[::-1])
    
    # 5. Generar puntos interpolados equidistantes
    length = sub_line.length
    if length <= 0 or num_points <= 1:
        pts_utm = [pt_start for _ in range(num_points)]
    else:
        distancias = np.linspace(0, length, num_points)
        pts_utm = [sub_line.interpolate(d) for d in distancias]
        
    # 6. Devolver a WGS84 (Lat/Lon)
    pts_wgs = []
    for pt in pts_utm:
        lon, lat = TRANSFORMER_TO_WGS.transform(pt.x, pt.y)
        pts_wgs.append(Point(lon, lat))
        
    return pts_wgs


import pandas as pd
import numpy as np
import networkx as nx
from shapely.geometry import Point, LineString

def complete_route(id, registros_person, 
                G_drive, G_walk,
                ig_drive, ig_walk,      
                map_drive, map_walk,    
                geometry_metro=None):
    """
    Realiza el map-matching con iGraph priorizando la continuidad geométrica para visualización.
    - Usa límites de velocidad por modo.
    - Aplica recálculo espacial estricto (d = v*t) si el GPS rebota pero mantiene la geometría.
    - Si falla el ruteo, guarda un POINT y no una recta atravesando edificios.
    - Valida estrictamente (sin buffer temporal) el reloj original.
    """
    rpc_list = [] 

    trips_arr = registros_person['trip'].to_numpy()
    lats_arr = registros_person['latitude'].to_numpy()
    lons_arr = registros_person['longitude'].to_numpy()
    
    nodes_drive_arr = registros_person['nodes_drive'].to_numpy() 
    nodes_walk_arr = registros_person['nodes_walk'].to_numpy()
    dis_arr = registros_person['dis lineal [m]'].to_numpy()
    timestamps_list = registros_person['local_timestamp'].tolist() 
    
    if 'modo_transporte' in registros_person.columns:
        modos_arr = registros_person['modo_transporte'].to_numpy()
    else:
        modos_arr = np.array(['Carro'] * len(registros_person))

    nodo_final_anterior = None
    trip_anterior = None
    
    # Limites estrictos de velocidad humana y física
    limites_kmh = {
        'Caminar': 15.0,'Bus': 100.0, 'Metro': 120.0, 'Carro': 150.0
    }

    for row in range(len(registros_person) - 1):
        trip_id = trips_arr[row] 
        modo_actual = modos_arr[row]
        dis_lineal_precalculada = dis_arr[row]
        max_speed_kmh = limites_kmh.get(modo_actual, 150.0)
        
        do_routing = False
        es_peaton = (str(modo_actual).lower() == 'caminar')
        
        # Selección dinámica de grafos
        G_actual = G_walk if es_peaton else G_drive
        ig_actual = ig_walk if es_peaton else ig_drive 
        map_actual = map_walk if es_peaton else map_drive 
        
        start_candidates = nodes_walk_arr[row] if es_peaton else nodes_drive_arr[row]
        end_candidates = nodes_walk_arr[row+1] if es_peaton else nodes_drive_arr[row+1]
        
        if not isinstance(start_candidates, (list, np.ndarray)): start_candidates = [start_candidates]
        if not isinstance(end_candidates, (list, np.ndarray)): end_candidates = [end_candidates]

        if trip_id == trip_anterior and nodo_final_anterior is not None:
            start_candidates = [nodo_final_anterior]
        elif trip_id != trip_anterior:
            nodo_final_anterior = None
            
        trip_anterior = trip_id

        if trip_id > 0:
            do_routing = True

        current_time = timestamps_list[row]
        best_route = None
        best_ratio = float('inf')
        time_calc_best = 0
        is_excessive_speed = False

        if do_routing:
            for u in start_candidates:
                for v in end_candidates:
                    if u == v: continue 
                    try:
                        ig_u, ig_v = map_actual[u], map_actual[v]
                        route_ig = ig_actual.get_shortest_paths(ig_u, to=ig_v, weights='length', output='vpath')[0]
                        
                        if len(route_ig) < 2: continue
                            
                        route = [ig_actual.vs[nx_id]['_nx_name'] for nx_id in route_ig]
                        route_length = sum(G_actual[route[i]][route[i+1]][0].get('length', 0) for i in range(len(route)-1))
                        
                        # --- VALIDACIÓN FÍSICA ESTRICTA (Reloj Puro) ---
                        time_real = (timestamps_list[row+1] - timestamps_list[row]).total_seconds()
                        speed_check = (route_length / 1000.0) / (time_real / 3600.0) if time_real > 0 else 0
                        
                        ratio = route_length / dis_lineal_precalculada if dis_lineal_precalculada > 0 else 1.0
                        score = abs(ratio - 1.0) 
                        
                        if score < best_ratio:
                            best_ratio = score
                            best_route = route
                            best_route_length = route_length
                            time_calc_best = sum(G_actual[route[i]][route[i+1]][0].get('travel_time', 0) for i in range(len(route)-1))
                            is_excessive_speed = (speed_check > max_speed_kmh)
                            
                        if score < 0.15: break 
                    except Exception: continue 
                if best_ratio < 0.15: break

        # ==========================================
        # 1. RUTA ENCONTRADA (Con o sin error de física)
        # ==========================================
        if best_route is not None:
            time_real = (timestamps_list[row+1] - timestamps_list[row]).total_seconds()
            
            for i in range(len(best_route)-1):
                u, v = best_route[i], best_route[i+1]
                edge_data = G_actual[u][v][0] 
                
                l_row = edge_data.get("length", 0)
                t_ideal = edge_data.get("travel_time", 1) 
                time_alloc = time_real * (t_ideal / time_calc_best) if time_calc_best > 0 else (time_real / (len(best_route)-1))
                
                speed_kph = (l_row / 1000.0) / (time_alloc / 3600.0) if time_alloc > 0 else 0
                
                flag_corregido = False


                if is_excessive_speed or speed_kph > max_speed_kmh:
                    if l_row <= 250.0: 
                        final_speed = max_speed_kmh
                        # Recálculo de distancia real (d = v * t)
                        final_dist = (max_speed_kmh * 1000.0 / 3600.0) * time_alloc
                        status_error = False
                        flag_corregido = True
                    else:
                        final_speed = 0.0
                        final_dist = 0.0
                        status_error = True
                else:
                    final_speed = speed_kph
                    final_dist = l_row
                    status_error = False

                if 'geometry' in edge_data: geom = edge_data['geometry'].wkt
                else: geom = f"LINESTRING ({G_actual.nodes[u]['x']} {G_actual.nodes[u]['y']}, {G_actual.nodes[v]['x']} {G_actual.nodes[v]['y']})"

                rpc_list.append({
                    'caid': id, 'trip': trip_id,    
                    'latitude': G_actual.nodes[u]['y'], 'longitude': G_actual.nodes[u]['x'],
                    'Speed [km/h]': final_speed,
                    'local_timestamp': current_time,
                    'start_node': u,
                    'end_node': v,
                    'osmid': str(edge_data.get("osmid", "N/A")),
                    'highway': edge_data.get("highway", "unclassified"), 
                    'geometry': geom, 'distance_m': final_dist,
                    'modo_transporte': modo_actual,
                    'ruteo_fallido': status_error,
                    'corregido_espacialmente': flag_corregido 
                })
                current_time = current_time + pd.Timedelta(seconds=time_alloc)
            nodo_final_anterior = best_route[-1]

        # ==========================================
        # 2. FALLO TOTAL O PARADA
        # ==========================================
        else:
            lat1, lon1 = lats_arr[row], lons_arr[row]
            current_time = timestamps_list[row]
            
            try:
                u_fallback = start_candidates[0]
                lon_u, lat_u = G_actual.nodes[u_fallback]['x'], G_actual.nodes[u_fallback]['y']
            except Exception:
                lon_u, lat_u = lon1, lat1

            if trip_id > 0:
                hw_type = 'routing_error'
                es_fallo = True 
            else:
                hw_type = 'parada_inactiva'
                es_fallo = False 

            geom = f'POINT ({lon_u} {lat_u})'

            rpc_list.append({
                'caid': id, 'trip': trip_id,         
                'latitude': lat_u if trip_id > 0 else lon1,  
                'longitude': lon_u if trip_id > 0 else lat1, 
                'Speed [km/h]': 0.0, 
                'local_timestamp': current_time,
                'start_node': start_candidates[0] if isinstance(start_candidates, list) else start_candidates, 
                'end_node': end_candidates[0] if isinstance(end_candidates, list) else end_candidates, 
                'osmid': 'N/A', 'highway': hw_type, 
                'geometry': geom, 'distance_m': 0.0, 
                'modo_transporte': modo_actual,
                'ruteo_fallido': es_fallo,
                'corregido_espacialmente': False 
            })
            
            nodo_final_anterior = None

    return pd.DataFrame(rpc_list)

## 2. Pipeline de Ejecución

#### 2.1. GPS Data

In [ ]:
# ==============================================================================
# 1. INGESTA Y SEGMENTACIÓN TEMPORAL
# ==============================================================================
print("Cargando datos GPS de usuarios...")
df = pd.read_parquet(FILE_GPS_ORIGINAL)

# --- ZONA HORARIA (UTC -> MONTERREY) ---
print("Ajustando reloj satelital a hora local de Monterrey...")
col_tiempo = 'utc_timestamp' if 'utc_timestamp' in df.columns else 'date'

try:
    df[col_tiempo] = pd.to_datetime(df[col_tiempo], unit='s')
except ValueError:
    df[col_tiempo] = pd.to_datetime(df[col_tiempo])

if df[col_tiempo].dt.tz is None:
    df[col_tiempo] = df[col_tiempo].dt.tz_localize('UTC').dt.tz_convert('America/Monterrey')
else:
    df[col_tiempo] = df[col_tiempo].dt.tz_convert('America/Monterrey')

df[col_tiempo] = df[col_tiempo].dt.tz_localize(None)
df = df.rename(columns={col_tiempo: 'local_timestamp'})
print(f"Reloj ajustado. Columna de tiempo actualizada a: 'local_timestamp'")

# --- FILTRO DE USUARIOS (MUESTRA) ---
num_users = 4 # Cambia esto para procesar más usuarios
usuarios_unicos = df['caid'].unique()[:num_users]
df = df[df['caid'].isin(usuarios_unicos)].copy()

pings_originales = len(df)
print(f"Registros originales antes de limpieza: {pings_originales:,}")

# --- PURIFICACIÓN DE LA SEÑAL (DOWNSAMPLING ESTABILIZADO) ---
print("Estandarizando la señal temporal (10s)...")
df = df.sort_values(by=['caid', 'local_timestamp'])

# Creamos ventana de 10 segundos
df['time_bucket'] = df['local_timestamp'].dt.floor('10s')

# Mantenemos el ÚLTIMO (keep='last') de cada ventana de 10s para mayor precisión
df = df.drop_duplicates(subset=['caid', 'time_bucket'], keep='last').copy()
df = df.drop(columns=['time_bucket'])

pings_limpios = len(df)
print(f"Registros sanos después del Downsampling: {pings_limpios:,}")
print(f"Ruido mitigado: {pings_originales - pings_limpios:,} ({(1 - pings_limpios/pings_originales)*100:.2f}%)")

# --- DELIMITACIÓN ESPACIAL (GEOFENCING) ---
ref_lat, ref_lon = 25.6866, -100.3161
threshold_distance = 48 # km
df['Distance'] = haversine_vectorized(ref_lat, ref_lon, df['latitude'].values, df['longitude'].values)
df = df[df['Distance'] < threshold_distance]

# Eliminar IDs con muy pocos datos
df = delete_ids_with_few_rows(df, df['caid'].unique(), threshold=2)

print(f"Total registros válidos post-filtrado: {len(df):,}")

# --- ESCUDO DE BORDES (DÍAS COMPLETOS) ---
df['date'] = df['local_timestamp'].dt.date
dias_unicos = sorted(df['date'].unique())

if len(dias_unicos) >= 3:
    print("Aplicando limpieza de bordes para conservar solo días 100% completos...")
    fecha_min, fecha_max = dias_unicos[0], dias_unicos[-1]
    df = df[(df['date'] > fecha_min) & (df['date'] < fecha_max)]
    print(f"Rango útil: {df['date'].min()} al {df['date'].max()}")
else:
    print("Protección de bordes omitida (pocos días en la muestra).")

# ==============================================================================
# 2. CÁLCULO DE VECTORES FÍSICOS (VELOCIDAD Y DISTANCIA)
# ==============================================================================
print("Calculando vectores de desplazamiento...")
df = df.sort_values(by=['caid', 'local_timestamp'])

df['lat_end'] = df.groupby(['caid', 'date'])['latitude'].shift(-1)
df['lon_end'] = df.groupby(['caid', 'date'])['longitude'].shift(-1)
df['local_timestamp_end'] = df.groupby(['caid', 'date'])['local_timestamp'].shift(-1)

df = df.dropna(subset=['lat_end', 'lon_end', 'local_timestamp_end'])

# Distancia en metros y tiempo en segundos
df['dis lineal [m]'] = haversine_vectorized(
    df['latitude'].values, df['longitude'].values, 
    df['lat_end'].values, df['lon_end'].values
) * 1000

df['travel time_sec'] = (df['local_timestamp_end'] - df['local_timestamp']).dt.total_seconds()

# Velocidad km/h
df['Speed [km/h]'] = np.where(
    df['travel time_sec'] > 0, 
    (df['dis lineal [m]'] / 1000.0) / (df['travel time_sec'] / 3600.0), 
    0
)

# Filtro de seguridad física (150 km/h)
df = df[df['Speed [km/h]'] <= 150]

# Limpieza de columnas auxiliares
df = df.drop(['lat_end', 'lon_end', 'local_timestamp_end', 'travel time_sec', 'Distance'], axis=1)
df = df.reset_index(drop=True)

print("Carga de GPS User Data completadoooo. :)")
df.head()

In [ ]:
import geopandas as gpd
from scipy.spatial import cKDTree
import numpy as np
import pickle
import osmnx as ox

# ==============================================================================
# 1. CARGA O PROYECCIÓN DE GRAFOS (INFRAESTRUCTURA)
# ==============================================================================
archivo_drive_proj = INFRA_DIR / 'G_drive_proyectado.pkl'
archivo_walk_proj = INFRA_DIR / 'G_walk_proyectado.pkl'

if archivo_drive_proj.exists() and archivo_walk_proj.exists():
    print("Cargando grafos proyectados desde INFRA_DIR...")
    with open(archivo_drive_proj, 'rb') as f:
        G_drive_proj = pickle.load(f)
    with open(archivo_walk_proj, 'rb') as f:
        G_walk_proj = pickle.load(f)
else:
    print("Proyectando grafos a metros (UTM)...")
    # G_drive y G_walk deben estar cargados en memoria
    G_drive_proj = ox.project_graph(G_drive)
    G_walk_proj = ox.project_graph(G_walk)
    
    print(f"Guardando proyecciones en {INFRA_DIR.name}...")
    with open(archivo_drive_proj, 'wb') as f:
        pickle.dump(G_drive_proj, f)
    with open(archivo_walk_proj, 'wb') as f:
        pickle.dump(G_walk_proj, f)

# ==============================================================================
# 2. ASIGNACIÓN ESPACIAL (MAP-MATCHING CON KD-TREE)
# ==============================================================================
print("Iniciando asignación espacial al grafo de la ciudad...")

# Proyectar los puntos del usuario a la misma zona UTM del grafo
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")
gdf_proj = gdf.to_crs(G_drive_proj.graph['crs'])

print("Construyendo árboles espaciales y buscando las 3 aristas candidatas...")

def get_k_nearest_edge_nodes(G_proj, gdf_points, k=3):
    """
    Encuentra las k aristas más cercanas a cada punto usando sus centroides.
    """
    _, edges_gdf = ox.graph_to_gdfs(G_proj)
    tree = cKDTree(np.c_[edges_gdf.centroid.x, edges_gdf.centroid.y])
    
    coords = np.c_[gdf_points.geometry.x, gdf_points.geometry.y]
    _, indices = tree.query(coords, k=k)
    
    if k == 1: 
        indices = indices.reshape(-1, 1)
        
    edge_uvs = np.array([idx[:2] for idx in edges_gdf.index]) 
    nodos_candidatos = []
    
    for i in range(len(coords)):
        aristas_ping = edge_uvs[indices[i]]
        nodos_unicos = list(set([nodo for arista in aristas_ping for nodo in arista]))
        nodos_candidatos.append(nodos_unicos)
        
    return nodos_candidatos

# Asignar los nodos candidatos
df['nodes_drive'] = get_k_nearest_edge_nodes(G_drive_proj, gdf_proj, k=3)
df['nodes_walk'] = get_k_nearest_edge_nodes(G_walk_proj, gdf_proj, k=3)

# --- LIMPIEZA DE RAM ---
# Ahora sí, borramos las proyecciones pesadas porque ya tenemos los nodos en el 'df'
del G_drive_proj, G_walk_proj, gdf, gdf_proj

print("Asignación espacial completada. 'nodes_drive' y 'nodes_walk' listos.")

### 2.2. Road Network Data

In [ ]:
import os
import osmnx as ox
import pickle
from pathlib import Path

if FILE_GRAFO.exists():
    print("Cargando grafo de AUTOS...")
    with open(FILE_GRAFO, 'rb') as f:
        G_drive = pickle.load(f)

    if FILE_GRAFO_WALK.exists():
        print("Cargando grafo de PEATONES...")
        with open(FILE_GRAFO_WALK, 'rb') as f:
            G_walk = pickle.load(f)
    else:
        print("IMPORTANT: Grafo de peatones no encontrado. Usando el de autos como temporal.")
        G_walk = G_drive
        
import igraph as ig

print("Convirtiendo grafos a iGraph...")
# Convertir grafos preservando el ID original de OSMnx en el atributo '_nx_name'
ig_drive = ig.Graph.from_networkx(G_drive)
ig_walk = ig.Graph.from_networkx(G_walk)

print("Creando diccionarios de indexación rápida...")
map_drive = {name: v.index for v, name in zip(ig_drive.vs, ig_drive.vs['_nx_name'])}
map_walk = {name: v.index for v, name in zip(ig_walk.vs, ig_walk.vs['_nx_name'])}
print("Infraestructura vial lista y optimizada.")

### 2.3. Completado de Rutas

In [ ]:
from joblib import Parallel, delayed
import pandas as pd
import numpy as np
import pickle
import traceback

# ==============================================================================
# 1. FUNCIÓN ENVOLTORIO (PIPELINE M1 -> M2 -> M3) CON DEBUG
# ==============================================================================
def process_day_wrapper(id_usuario, fecha, df_dia):
    """
    Procesa un día de un usuario: Segmenta, Clasifica y Rutea.
    """
    if isinstance(df_dia, pd.Series):
        df_dia = df_dia.to_frame().T

    df_dia = df_dia.reset_index(drop=True)

    if df_dia.empty:
        return pd.DataFrame() 

    try:
        # --- PASO 1: SEGMENTACIÓN (Módulo 1) ---
        df_dia = assign_trips(df_dia)
        
        # --- PASO 2: CLASIFICACIÓN MODAL (Módulo 2) ---
        if 'trip' in df_dia.columns:  
            df_dia = calcular_cercania_infraestructura(df_dia, subway_routes, bus_routes)
            df_dia = clasificar_viajes(df_dia)
        else:
            df_dia['modo_transporte'] = 'Carro' 
            
        # --- PASO 3: COMPLETADO DE RUTAS (Módulo 3 - Híbrido iGraph) ---
        df_routed = complete_route(
            id=id_usuario, 
            registros_person=df_dia, 
            G_drive=G_drive, 
            G_walk=G_walk,
            ig_drive=ig_drive,     
            ig_walk=ig_walk,       
            map_drive=map_drive,   
            map_walk=map_walk,     
            geometry_metro=subway_routes_proj
        )
        return df_routed

    except Exception as e:
        # DEBUG PARA MAC: Si falla, imprimimos el error exacto y el lugar
        print(f"Error crítico en Usuario {id_usuario} (Fecha: {fecha}):")
        print(e)
        return pd.DataFrame()

# ==============================================================================
# 2. EJECUCIÓN PARALELA CORREGIDA
# ==============================================================================
# 0. Asegurar columna fecha
df['local_timestamp'] = pd.to_datetime(df['local_timestamp'])
df['date'] = df['local_timestamp'].dt.date

# PARCHE: Regenerar la columna 'travel time' para assign_trips
df['travel time'] = df.groupby(['caid', 'date'])['local_timestamp'].shift(-1) - df['local_timestamp']
df['travel time'] = df['travel time'].fillna(pd.Timedelta(seconds=0)) # Protegemos los nulos del final del día

# Generar lista de tareas
grupos = list(df.groupby(['caid', 'date']))
print(f"Iniciando procesamiento paralelo de {len(grupos)} tareas...")

# Ejecución paralela
resultados = Parallel(n_jobs=7, backend="loky", verbose=10)(
    delayed(process_day_wrapper)(
        id_usuario, fecha_dia, grupo_df
    ) for (id_usuario, fecha_dia), grupo_df in grupos
)

# ==============================================================================
# 3. CONSOLIDACIÓN DE RESULTADOS (BLINDADA)
# ==============================================================================
print("Agregando resultados y consolidando topología...")

# Filtramos los que no están vacíos
results_clean = [r for r in resultados if r is not None and not r.empty] 

if len(results_clean) > 0:
    df_completed_routes = pd.concat(results_clean, axis=0, ignore_index=True)

    # Limpieza final de columnas y tipos
    if 'date' in df_completed_routes.columns:
        df_completed_routes = df_completed_routes.drop(columns=['date'])

    df_completed_routes['caid'] = df_completed_routes['caid'].astype(str)
    df_completed_routes['trip'] = df_completed_routes['trip'].astype(int)

    df_completed_routes = df_completed_routes.reset_index(drop=True)
    print(f"Ruteo finalizado. Se generaron {len(df_completed_routes):,} puntos de ruta.")
else:
    print("ERROR: No se generó ningún resultado. Revisa los errores impresos arriba.")
    df_completed_routes = pd.DataFrame()

#### Module 2 Sanity Check

In [ ]:
# ==============================================================================
# SANITY CHECK: MÓDULOS 1 Y 2 (MODOS Y RUTEO) 
# ==============================================================================
print("="*60)
print("REPORTE DE DIAGNÓSTICO: RUTAS Y MODOS")
print("="*60)

total_registros = len(df_completed_routes)
print(f"▶ Total de puntos generados: {total_registros:,}")

print("\n▶ 1A. Distribución Modal (Por cantidad de registros):")
modos_pct_regs = df_completed_routes['modo_transporte'].value_counts(normalize=True) * 100
print(modos_pct_regs.round(2).astype(str) + " %")

print("\n▶ 1B. Distribución Modal (Por distancia viajada):")
distancia_por_modo = df_completed_routes.groupby('modo_transporte')['distance_m'].sum()
distancia_pct = (distancia_por_modo / distancia_por_modo.sum()) * 100
print(distancia_pct.sort_values(ascending=False).round(2).astype(str) + " %")

print("\n▶ 2. Distribución de vías (Top 5 tipos):")
vias_pct = df_completed_routes['highway'].value_counts(normalize=True) * 100
print(vias_pct.head(5).round(2).astype(str) + " %")

# ---------------------------------------------------------
# NUEVA SECCIÓN DE DIAGNÓSTICO DE RUTEO
# ---------------------------------------------------------
print("\n▶ 3. Tasa de Éxito en Completado de Rutas:")
if 'ruteo_fallido' in df_completed_routes.columns:
    # Aislamos solo los puntos en movimiento (ignoramos paradas)
    df_viajes = df_completed_routes[df_completed_routes['trip'] > 0]
    total_viajes = len(df_viajes)
    
    fallos_totales = df_viajes['ruteo_fallido'].sum()
    
    if fallos_totales > 0:
        pct_errores = (fallos_totales / total_viajes) * 100
        exito_pct = 100 - pct_errores
        print(f"  Tasa de éxito general (puros): {exito_pct:.2f}%")
        print(f"  ADVERTENCIA: {fallos_totales:,} segmentos ({pct_errores:.2f}%) fallaron el map-matching y fueron anulados.")
        
        # Desglose del tipo de error
        fallos_topologia = (df_viajes['highway'] == 'routing_error').sum()
        fallos_fisica = fallos_totales - fallos_topologia
        
        print(f"      ↳ {fallos_fisica:,} anulados por salto masivo de GPS (Teletransporte)")
        print(f"      ↳ {fallos_topologia:,} anulados por desconexión en el mapa (Topología)")
    else:
        print("  ÉXITO: 100% de los segmentos ruteados correctamente. 0 fallos.")
else:
    print("  No se encontró la columna 'ruteo_fallido' para evaluar el completado.")

# ---------------------------------------------------------
# NUEVA SECCIÓN DE AUDITORÍA DE CORRECCIÓN ESPACIAL
# ---------------------------------------------------------
print("\n▶ 4. Auditoría de Filtro Anti-Rebotes (Corrección Espacial):")
if 'corregido_espacialmente' in df_completed_routes.columns:
    corregidos = df_viajes['corregido_espacialmente'].sum()
    if total_viajes > 0:
        pct_corregidos = (corregidos / total_viajes) * 100
        print(f"   {corregidos:,} segmentos ({pct_corregidos:.2f}%) tenían exceso de velocidad y fueron rescatados.")
        print("      (Se mantuvo su dibujo en el mapa, pero se recalculó su distancia real)")
    else:
        print("  No hay viajes registrados para evaluar.")
else:
    print("  No se encontró la bandera de 'corregido_espacialmente'.")

print("="*60)

### 2.4. Guardado de Completado de Rutas

In [ ]:
# ==============================================================================
# 5. LIMPIEZA Y GUARDADO INTELIGENTE A PARQUET
# ==============================================================================
print("Guardando rutas completadas...")

# 1. Quitar las columnas temporales del ruteo espacial
df_completed_routes = df_completed_routes.drop(columns=["nodes_drive", "nodes_walk", "date"], errors="ignore")

# 2. LA APLANADORA DE TIPOS (Directo en memoria RAM)
print("Estandarizando tipos de datos para Apache Arrow...")

# Forzar columnas de Texto (Strings puros)
cols_texto = ['caid', 'start_node', 'end_node', 'osmid', 'highway', 'geometry', 'modo_transporte']
for col in cols_texto:
    if col in df_completed_routes.columns:
        df_completed_routes[col] = df_completed_routes[col].astype(str)

# Forzar columnas Numéricas (Enteros y Floats)
if 'trip' in df_completed_routes.columns:
    df_completed_routes['trip'] = df_completed_routes['trip'].fillna(-1).astype(int)
    
cols_float = ['latitude', 'longitude', 'Speed [km/h]', 'distance_m']
for col in cols_float:
    if col in df_completed_routes.columns:
        df_completed_routes[col] = pd.to_numeric(df_completed_routes[col], errors='coerce').astype(float)

# Forzar columna de Fecha/Hora
if 'local_timestamp' in df_completed_routes.columns:
    df_completed_routes['local_timestamp'] = pd.to_datetime(df_completed_routes['local_timestamp'], errors='coerce')
    
# Forzar Booleanos 
if 'ruteo_fallido' in df_completed_routes.columns:
    df_completed_routes['ruteo_fallido'] = df_completed_routes['ruteo_fallido'].fillna(False).astype(bool)

if 'corregido_espacialmente' in df_completed_routes.columns:
    df_completed_routes['corregido_espacialmente'] = df_completed_routes['corregido_espacialmente'].fillna(False).astype(bool)

# 3. GUARDADO FINAL (USANDO PATHLIB)
print(f"Guardando archivo en formato Parquet en la carpeta '{INTERMEDIATE_DIR.name}'...")

# Usamos la variable inteligente de tu bloque de configuración
df_completed_routes.to_parquet(FILE_INTERMEDIO, index=False)

print("Ruteo finalizado y guardado con éxito.")

In [ ]:
import pandas as pd

# 1. Aislamos exclusivamente el % sintético
df_corregidos = df_completed_routes[df_completed_routes['corregido_espacialmente'] == True].copy()

print("============================================================")
print("AUDITORÍA DE DATOS REALES Y SINTÉTICOS")
print("============================================================")
print(f"Total de segmentos analizados: {len(df_corregidos):,}\n")

# 2. ¿De qué tamaño es la distancia que les asignamos (la sintética)?
print("▶ 1. DISTANCIA SINTÉTICA ASIGNADA (Metros):")
print(df_corregidos['distance_m'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]).round(2))

# 3. ¿A qué modos de transporte les está pasando más esto?
print("\n▶ 2. MODOS DE TRANSPORTE AFECTADOS:")
modos_afectados = df_corregidos['modo_transporte'].value_counts(normalize=True) * 100
print(modos_afectados.round(2).astype(str) + " %")

# 4. ¿Qué tipo de calles están provocando estos errores?
print("\n▶ 3. TIPO DE VIALIDAD DONDE OCURRE EL REBOTE:")
vias_afectadas = df_corregidos['highway'].value_counts(normalize=True) * 100
print(vias_afectadas.head(5).round(2).astype(str) + " %")

# ============================================================
# 5. EXPORTACIÓN PARA INSPECCIÓN VISUAL (KEPLER)
# ============================================================
ruta_muestra_visual = INTERMEDIATE_DIR / "Muestra_Tramos_Corregidos_Kepler.csv"

# Guardamos solo una muestra representativa (ej. 1000 tramos) para no saturar Kepler
df_corregidos.sample(n=min(1000, len(df_corregidos)), random_state=42).to_csv(ruta_muestra_visual, index=False)

print("\n============================================================")
print(f"ARCHIVO CREADO: {ruta_muestra_visual.name}")
print("============================================================")

In [ ]:
import pandas as pd

print("============================================================")
print("BALANCE DE DISTANCIA TOTAL: PURO VS. SINTÉTICO")
print("============================================================")

# 1. Filtramos solo los viajes (ignoramos las paradas que tienen distancia 0)
df_viajes = df_completed_routes[df_completed_routes['trip'] > 0].copy()

# 2. Sumamos las distancias (convertimos de metros a kilómetros)
dist_pura_km = df_viajes[df_viajes['corregido_espacialmente'] == False]['distance_m'].sum() / 1000.0
dist_sintetica_km = df_viajes[df_viajes['corregido_espacialmente'] == True]['distance_m'].sum() / 1000.0

dist_total_km = dist_pura_km + dist_sintetica_km

if dist_total_km > 0:
    pct_pura = (dist_pura_km / dist_total_km) * 100
    pct_sintetica = (dist_sintetica_km / dist_total_km) * 100

    print(f"▶ DISTANCIA TOTAL RUTEADA: {dist_total_km:,.2f} km\n")
    
    print(f"  Ruteo Puro (Intacto):      {dist_pura_km:,.2f} km ({pct_pura:.2f}%)")
    print(f"  Ruteo Sintético (Topado):   {dist_sintetica_km:,.2f} km ({pct_sintetica:.2f}%)")
else:
    print("No hay distancia registrada en los viajes.")

print("============================================================")

# **MODULE 3: Código de Cálculo de Emisiones**

## Settings y Cálculo Dinámico 

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
print("Iniciando Módulo 3: Cálculo de emisiones.")

COLUMNA_MODO = 'modo_transporte' 
COLUMNA_VELOCIDAD = 'Speed [km/h]'
COLUMNA_DISTANCIA = 'distance_m'

print("Cargando rutas y tasas de emisión...")

# Cargamos el archivo puente del Módulo 2
df_rutas = pd.read_parquet(FILE_INTERMEDIO)

# Cargamos la tabla de MOVES
df_emisiones_raw = pd.read_parquet(FILE_MOVES)

# INCLUSIÓN DE TODOS LOS CONTAMINANTES CRITERIO
POLLUTANTS = ['CO', 'CO2', 'CO2_Equiv', 'HC', 'NOx', 'PM10', 'PM25']

OSM_TO_MOVES_ROAD = {
    'motorway': 4, 'motorway_link': 4, 'trunk': 4, 'trunk_link': 4, 
    'primary': 4, 'primary_link': 4, 'secondary': 5, 'secondary_link': 5, 
    'tertiary': 5, 'tertiary_link': 5, 'unclassified': 5, 'residential': 5,
    'routing_error': 5 
}

# ==============================================================================
# 2. PREPARACIÓN DE VARIABLES
# ==============================================================================
print("Preparando variables temporales y espaciales...")
df_rutas['orden_original'] = range(len(df_rutas))

df_rutas['local_timestamp'] = pd.to_datetime(df_rutas['local_timestamp'])

# Extracción de Mes, Hora y Día
df_rutas['Month'] = df_rutas['local_timestamp'].dt.month
df_rutas['Hour'] = df_rutas['local_timestamp'].dt.hour + 1
df_rutas['Day'] = np.where(df_rutas['local_timestamp'].dt.dayofweek < 5, 5, 2)
    
df_rutas['Road'] = df_rutas['highway'].map(OSM_TO_MOVES_ROAD).fillna(5).astype(int)

# Clasificación de Source ID
condiciones = [
    df_rutas['trip'] < 0,
    df_rutas[COLUMNA_MODO].astype(str).str.contains('(?i)carro|auto', na=False),
    df_rutas[COLUMNA_MODO].astype(str).str.contains('(?i)bus', na=False)
]
opciones = [0, 21, 42]
df_rutas['Source'] = np.select(condiciones, opciones, default=0)

# SpeedBins
df_rutas['avg_speed_mph'] = df_rutas[COLUMNA_VELOCIDAD] * 0.621371
bins = [0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, float('inf')]
df_rutas['SpeedBin'] = pd.cut(df_rutas['avg_speed_mph'], bins=bins, labels=list(range(1, 17)), right=False)
df_rutas['SpeedBin'] = df_rutas['SpeedBin'].astype(object).fillna(1).astype(int)

# ==============================================================================
# 3. CRUCE CON MOVES EMISSION RATES
# ==============================================================================
print("Cruzando con factores de emisión...")

merge_cols = ['Day', 'Hour', 'Road', 'Source', 'SpeedBin']
cols_needed = merge_cols + POLLUTANTS
df_emisiones_raw = df_emisiones_raw[cols_needed].astype({
    'Day': int, 'Hour': int, 'Road': int, 'Source': int, 'SpeedBin': int
})

# Optimización de la tabla de emisiones
df_emisiones = df_emisiones_raw.groupby(merge_cols, as_index=False)[POLLUTANTS].mean()

df_motorizados = df_rutas[df_rutas['Source'] > 0].copy()
df_no_motorizados = df_rutas[df_rutas['Source'] == 0].copy()

# NIVEL 1: Cruce Exacto
df_motorizados = pd.merge(df_motorizados, df_emisiones, on=merge_cols, how='left')

mask_faltantes = df_motorizados[POLLUTANTS[0]].isna()
if mask_faltantes.any():
    print(f"▶ Imputando {mask_faltantes.sum():,} registros sin cruce exacto...")
    
    promedios_sb = df_emisiones.groupby(['Source', 'SpeedBin'])[POLLUTANTS].mean().reset_index()
    promedios_source = df_emisiones.groupby('Source')[POLLUTANTS].mean().reset_index()
    
    # NIVEL 2: Imputación por SpeedBin (Vecino cercano)
    fuentes_unicas = df_emisiones['Source'].unique()
    idx_completo = pd.MultiIndex.from_product([fuentes_unicas, range(1, 17)], names=['Source', 'SpeedBin'])
    df_curvas = promedios_sb.set_index(['Source', 'SpeedBin']).reindex(idx_completo).groupby(level='Source').ffill().groupby(level='Source').bfill().reset_index()
    
    df_curvas = df_curvas.rename(columns={p: f"{p}_curve" for p in POLLUTANTS})
    df_motorizados = pd.merge(df_motorizados, df_curvas, on=['Source', 'SpeedBin'], how='left')
    
    df_global = promedios_source.rename(columns={p: f"{p}_global" for p in POLLUTANTS})
    df_motorizados = pd.merge(df_motorizados, df_global, on='Source', how='left')
    
    # Aplicar cascada completa
    for p in POLLUTANTS:
        df_motorizados[p] = df_motorizados[p].fillna(df_motorizados[f"{p}_curve"])
        df_motorizados[p] = df_motorizados[p].fillna(df_motorizados[f"{p}_global"])
        df_motorizados[p] = df_motorizados[p].fillna(0.0) # Fallback final extremo
        
    df_motorizados = df_motorizados.drop(columns=[f"{p}_curve" for p in POLLUTANTS] + [f"{p}_global" for p in POLLUTANTS])

# Unir y limpiar
for p in POLLUTANTS: df_no_motorizados[p] = 0.0 
df_final = pd.concat([df_motorizados, df_no_motorizados], ignore_index=True).sort_values('orden_original').reset_index(drop=True)

# ==============================================================================
# 4. CÁLCULO DE TOTALES
# ==============================================================================
print("Calculando masa total de emisiones...")
df_final['distance_km_calc'] = df_final['distance_m'] / 1000.0

for p in POLLUTANTS:
    df_final[f"Densidad_{p}_g_km"] = df_final[p] 
    df_final[f"Total_{p}_g"] = df_final[f"Densidad_{p}_g_km"] * df_final['distance_km_calc']

df_final['fecha_kepler'] = df_final['local_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

print("Módulo 3 completado.")

#### Module 3 Sanity Check

In [ ]:
# ==============================================================================
# SANITY CHECK: MÓDULO 3 (EMISIONES MOVES)
# ==============================================================================
print("="*60)
print("REPORTE DE DIAGNÓSTICO: INVENTARIO DE EMISIONES")
print("="*60)

print("▶ 1. Mapeo de Identificadores (Source ID):")
resumen_source = df_final.groupby('Source').agg(
    Registros=('Source', 'size'),
    Distancia_km=('distance_km_calc', 'sum')
)
total_regs = resumen_source['Registros'].sum()
total_dist = resumen_source['Distancia_km'].sum()

for source, row in resumen_source.iterrows():
    pct_reg = (row['Registros'] / total_regs) * 100
    pct_dist = (row['Distancia_km'] / total_dist) * 100 if total_dist > 0 else 0
    nombre = 'No Motorizado/Parada' if source == 0 else ('Carro' if source == 21 else 'Bus')
    print(f"   - {nombre} (ID {source}): {row['Registros']:,} registros ({pct_reg:.2f}%) | {row['Distancia_km']:,.2f} km ({pct_dist:.2f}%)")

POLLUTANTS_noeqiv = ['CO', 'CO2', 'HC', 'NOx', 'PM10', 'PM25']

print("\n▶ 2. Inventario Total de Contaminantes (Global):")
totales_emisiones = {p: df_final[f"Total_{p}_g"].sum() for p in POLLUTANTS_noeqiv} 
serie_totales = pd.Series(totales_emisiones).sort_values(ascending=False)
total_masa = serie_totales.sum()

for p, masa in serie_totales.items():
    pct = (masa / total_masa) * 100 if total_masa > 0 else 0
    print(f"   - {p}: {masa:,.2f} g ({pct:.4f}%)")

print("\n▶ 3. Inventario Local de Calidad del Aire (EXCLUYENDO CO2 y CO2_Equiv):")
pol_excluidos = [p for p in POLLUTANTS if p not in ['CO2', 'CO2_Equiv']]
totales_sin_co2 = {p: df_final[f"Total_{p}_g"].sum() for p in pol_excluidos}
serie_sin_co2 = pd.Series(totales_sin_co2).sort_values(ascending=False)
total_masa_sin_co2 = serie_sin_co2.sum()

for p, masa in serie_sin_co2.items():
    pct = (masa / total_masa_sin_co2) * 100 if total_masa_sin_co2 > 0 else 0
    print(f"   - {p}: {masa:,.2f} g ({pct:.2f}%)")

print("\n▶ 4. Calidad del Cruce con MOVES (Efectividad del Modelo):")
# Filtramos solo los que usan motor (Carro, Bus)
motorizados = df_final[df_final['Source'] > 0]
total_motorizados = len(motorizados)

ceros_injustificados = motorizados[motorizados['Densidad_CO2_Equiv_g_km'] == 0]
segmentos_cero_dist = motorizados[motorizados['distance_m'] == 0]
fallos = len(ceros_injustificados)

if total_motorizados > 0:
    tasa_exito = ((total_motorizados - fallos) / total_motorizados) * 100
    print(f"   - Registros motorizados procesados: {total_motorizados:,}")
    print(f"   - Registros con factor de emisión asignado: {total_motorizados - fallos:,}")
    print(f"   - TASA DE ÉXITO DE MAPEO: {tasa_exito:.2f} %")
    
    if fallos > 0:
        print(f"   - Registros huérfanos (sin match en MOVES): {fallos:,} ({(fallos/total_motorizados)*100:.2f}%)")
else:
    print("   - No hay registros motorizados para evaluar.")

print(f"\n   Nota: Hay {len(segmentos_cero_dist):,} segmentos motorizados donde la distancia avanzada fue 0 metros.")
print("="*60)

## Guardado de Resultados Finales

#### In Parquet format

In [ ]:
df_final.to_parquet(FILE_FINAL_PARQUET)
print(f"Proceso terminado. Archivo de emisiones guardado en: {FILE_FINAL_PARQUET.name}")

#### In GeoJSON Format

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

print("Exportando mapa a GeoJSON para Kepler...")

# 1. Cargar el Parquet Maestro
df_mapa = pd.read_parquet(FILE_FINAL_PARQUET)

print("Seleccionando columnas para Kepler...")

# 2. Seleccionar SOLO las columnas que importan para visualizar 
columnas_mantener = ['caid', 'trip', 'fecha_kepler', 'Speed [km/h]', 'osmid', 'highway',
       'geometry', 'distance_m', 'modo_transporte', 'Densidad_CO_g_km', 'Total_CO_g', 'Densidad_CO2_g_km', 'Total_CO2_g',
       'Densidad_CO2_Equiv_g_km', 'Total_CO2_Equiv_g', 'Densidad_HC_g_km',
       'Total_HC_g', 'Densidad_NOx_g_km', 'Total_NOx_g', 'Densidad_PM10_g_km',
       'Total_PM10_g', 'Densidad_PM25_g_km', 'Total_PM25_g', 'latitude', 'longitude', 'start_node', 'end_node']

columnas_presentes = [c for c in columnas_mantener if c in df_mapa.columns]
df_mapa = df_mapa[columnas_presentes].copy()

# Remover filas donde no haya geometría
df_mapa = df_mapa.dropna(subset=['geometry'])

# ==============================================================================
# Convertir identificadores gigantes a texto para que el exportador C no colapse
columnas_gigantes = ['osmid', 'start_node', 'end_node', 'caid']
for col in columnas_gigantes:
    if col in df_mapa.columns:
        df_mapa[col] = df_mapa[col].astype(str)
# ==============================================================================

print(f"Construyendo el GeoJSON para {len(df_mapa)} segmentos (Optimizado en C)...")

# 3. Convertir los textos WKT a geometrías matemáticas reales y crear GeoDataFrame
geometrias = df_mapa['geometry'].apply(wkt.loads)
gdf_export = gpd.GeoDataFrame(df_mapa, geometry=geometrias, crs="EPSG:4326")

# 4. Exportar directo usando la variable global de la carpeta Outputs
print("Escribiendo archivo en disco duro...")
gdf_export.to_file(FILE_FINAL_GEOJSON, driver='GeoJSON')

# (Nota: corregí la variable del print final para que coincida con la que usaste al guardar)
print(f"Archivo GeoJSON guardado exitosamente en:\n'{FILE_FINAL_GEOJSON}'")

####    In CSV Format

In [ ]:
import pandas as pd

print("Exportando mapa a CSV para Kepler...")

df_mapa = pd.read_parquet(FILE_FINAL_PARQUET)

print("Seleccionando columnas para Kepler...")

columnas_mantener = ['caid', 'trip', 'fecha_kepler', 'Speed [km/h]', 'osmid', 'highway',
       'geometry', 'distance_m', 'modo_transporte', 'Densidad_CO_g_km', 'Total_CO_g', 'Densidad_CO2_g_km', 'Total_CO2_g',
       'Densidad_CO2_Equiv_g_km', 'Total_CO2_Equiv_g', 'Densidad_HC_g_km',
       'Total_HC_g', 'Densidad_NOx_g_km', 'Total_NOx_g', 'Densidad_PM10_g_km',
       'Total_PM10_g', 'Densidad_PM25_g_km', 'Total_PM25_g', 'latitude', 'longitude']

columnas_presentes = [c for c in columnas_mantener if c in df_mapa.columns]
df_mapa = df_mapa[columnas_presentes].copy()
df_mapa = df_mapa.dropna(subset=['geometry'])

print(f"Escribiendo {len(df_mapa)} segmentos a CSV...")
df_mapa.to_csv(FILE_FINAL_CSV, index=False) 

print(f"Archivo CSV guardado exitosamente en:\n'{FILE_FINAL_CSV}'")

### Results Visualization Dashboards

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ==============================================================================
# 5. DASHBOARD CIENTÍFICO: ANÁLISIS VISUAL DE RESULTADOS
# ==============================================================================
print("Generando Dashboard de Análisis Visual...")

# 1. Cargar datos
df_final = pd.read_parquet(FILE_FINAL_PARQUET)

# Configuración de estilo visual (Estilo académico y limpio)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Análisis de Resultados - GPS Data Emissions', fontsize=20, weight='bold', y=1.02)

# --- GRÁFICA 1: Movilidad (Tiempo vs Distancia) ---
# Preparamos los datos
resumen_modo = df_final.groupby('Source').agg(
    Registros=('Source', 'size'),
    Distancia_km=('distance_km_calc', 'sum')
)

# Normalizamos a porcentajes (Nombres simplificados para evitar KeyErrors)
resumen_modo['% Registros'] = (resumen_modo['Registros'] / resumen_modo['Registros'].sum()) * 100
resumen_modo['% Distancia Real'] = (resumen_modo['Distancia_km'] / resumen_modo['Distancia_km'].sum()) * 100

# Mapeo de nombres de Source
nombres_source = {0: 'No Motorizado', 21: 'Carro', 42: 'Bus'}
resumen_modo.index = resumen_modo.index.map(nombres_source).fillna('Otro')

# Graficamos con los nombres de columna EXACTOS
resumen_modo[['% Registros', '% Distancia Real']].plot(kind='bar', ax=axes[0, 0], color=['#3498db', '#2ecc71'])
axes[0, 0].set_title('Uso de Modos de Transporte', fontsize=14, weight='bold')
axes[0, 0].set_ylabel('Porcentaje (%)')
axes[0, 0].set_xlabel('')
axes[0, 0].tick_params(axis='x', rotation=0)

# --- GRÁFICA 2: Inventario de Tóxicos Locales (Sin CO2) ---
pol_locales = ['CO', 'NOx', 'HC', 'PM10', 'PM25']
# Extraer sumas y convertir a Kilogramos
masas_locales = [df_final[f"Total_{p}_g"].sum() / 1000 for p in pol_locales] 

sns.barplot(x=pol_locales, y=masas_locales, ax=axes[0, 1], palette="magma")
axes[0, 1].set_title('Inventario de Contaminantes Locales (Sin CO2)', fontsize=14, weight='bold')
axes[0, 1].set_ylabel('Masa Total Emitida (Kilogramos)')
axes[0, 1].set_xlabel('Contaminante')

# --- GRÁFICA 3: Dinámica Temporal ---
# Sumamos el CO2 por hora
if 'Hour' in df_final.columns:
    emisiones_hora = df_final[df_final['Source'] > 0].groupby('Hour')['Total_CO2_Equiv_g'].sum() / 1000 # En kg
    horas_completas = range(1, 25)
    emisiones_hora = emisiones_hora.reindex(horas_completas, fill_value=0)

    sns.lineplot(x=emisiones_hora.index, y=emisiones_hora.values, ax=axes[1, 0], color='#e74c3c', marker='o', linewidth=2.5)
    axes[1, 0].set_title('Perfil Diurno de Emisiones (CO2)', fontsize=14, weight='bold')
    axes[1, 0].set_ylabel('CO2 Equivalente Emitido (kg)')
    axes[1, 0].set_xlabel('Hora del Día (1-24)')
    axes[1, 0].set_xticks(range(1, 25, 2))

# --- GRÁFICA 4: Validación Científica (Curva de Emisión vs Velocidad) ---
if 'SpeedBin' in df_final.columns:
    df_motorizado = df_final[df_final['Source'] > 0]
    curva_velocidad = df_motorizado.groupby('SpeedBin')['Densidad_CO2_Equiv_g_km'].mean()

    sns.lineplot(x=curva_velocidad.index, y=curva_velocidad.values, ax=axes[1, 1], color='#9b59b6', marker='s', linewidth=2.5)
    axes[1, 1].set_title('Curva de Eficiencia: Emisión vs. Velocidad', fontsize=14, weight='bold')
    axes[1, 1].set_ylabel('Densidad Promedio (g CO2 / km)')
    axes[1, 1].set_xlabel('SpeedBin (Rangos de Velocidad MOVES)')
    axes[1, 1].set_xticks(range(1, 17))

# Ajuste final
plt.tight_layout()

# Guardado automático
archivo_grafica = FINAL_DIR / "Dashboard_Emisiones_Resultados.png"
# plt.savefig(archivo_grafica, dpi=300, bbox_inches='tight')
print(f"Dashboard generado.")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("Preparando auditoría visual de velocidades...")

df_viajes = pd.read_parquet(FILE_INTERMEDIO)

# 2. CONFIGURACIÓN DEL GRÁFICO
plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")

# 3. CREACIÓN DEL BOX PLOT
ax = sns.boxplot(
    data=df_viajes, 
    x='modo_transporte', 
    y='Speed [km/h]', 
    palette='Set2',
    linewidth=1.5,
    showfliers=True, 
    flierprops=dict(marker='o', color='red', alpha=0.3, markersize=4) 
)

# 4. FORMATO Y ETIQUETAS
plt.title('Auditoría de Map-Matching: Distribución de Velocidades por Modo\n(Excluyendo Paradas y Tramos sin Calle/OSMID)', 
          fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Modo de Transporte Asignado', fontsize=13, labelpad=10)
plt.ylabel('Velocidad Interpolada [km/h]', fontsize=13, labelpad=10)

# Ajustar el eje Y de forma dinámica basado en los datos filtrados
max_speed = int(df_viajes['Speed [km/h]'].max()) if not df_viajes.empty else 150
plt.yticks(range(0, max_speed + 20, 20))

plt.tight_layout()
plt.show()

# ==============================================================================
# 5. REPORTE ESTADÍSTICO EN CONSOLA
# ==============================================================================
print("\n" + "="*65)
print(" RESUMEN ESTADÍSTICO DE VELOCIDADES POR MODO [km/h]")
print("="*65)

if not df_viajes.empty:
    resumen = df_viajes.groupby('modo_transporte')['Speed [km/h]'].describe()
    resumen_limpio = resumen[['count', 'mean', 'std', 'min', '50%', '75%', 'max']].round(2)
    resumen_limpio = resumen_limpio.rename(columns={'50%': 'mediana', 'count': 'segmentos_validos'})
    print(resumen_limpio)
else:
    print("No hay datos disponibles con estos filtros.")
print("="*65)

### Reduced CSVs

Light

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Las columnas exactas para visualizar en Kepler
columnas_mantener = [
    'caid', 'fecha_kepler', 'Speed [km/h]',
    'geometry', 'distance_m', 'modo_transporte', 'Total_CO2_g'
]

print("Cargando el archivo Parquet original (leyendo solo las columnas indicadas)...")

# ALERTA DE OPTIMIZACIÓN: 'columns' hace que el disco duro solo lea esas columnas específicas, ahorrando RAM
df_ligero = pd.read_parquet(FILE_FINAL_PARQUET, columns=columnas_mantener)

# 2. Definir el nombre del nuevo archivo
archivo_nuevo = FILE_FINAL_PARQUET.parent / "Resultados_Mapa_Emisiones_GPS_Kepler_Light.csv"

print(f"Escribiendo el nuevo archivo CSV de {len(df_ligero)} filas...")
df_ligero.to_csv(archivo_nuevo, index=False)

print(f"Archivo reducido guardado exitosamente en:\n{archivo_nuevo}")

Super Light

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Las columnas exactas (AGREGAMOS 'highway' PARA PODER FILTRAR LOS ERRORES)
columnas_mantener = [
    'caid', 'fecha_kepler', 'Speed [km/h]', 'highway',
    'geometry', 'distance_m', 'modo_transporte', 'Total_CO2_g'
]

print("Cargando el archivo original desde Parquet...")
# ALERTA DE OPTIMIZACIÓN: Leemos directo del Parquet solo las columnas necesarias
df_ligero = pd.read_parquet(FILE_FINAL_PARQUET, columns=columnas_mantener)

# ==============================================================================
# 2. FILTRO ESTÉTICO (OPCIÓN B: Ocultar saltos en el mapa)
# ==============================================================================
registros_antes = len(df_ligero)
# Nos quedamos solo con las calles reales, borrando las líneas rectas de error
df_ligero = df_ligero[df_ligero['highway'] != 'routing_error']
registros_despues = len(df_ligero)

print(f"Filtro estético aplicado: Se ocultaron {registros_antes - registros_despues:,} saltos de 'routing_error'.")

# ==============================================================================
# 3. GUARDADO FINAL
# ==============================================================================
# Usamos el .parent de tu variable global FILE_FINAL_PARQUET para guardarlo en la misma carpeta
archivo_nuevo = FILE_FINAL_PARQUET.parent / "Resultados_Mapa_Emisiones_GPS_Kepler_SuperLight.csv"

print(f"Escribiendo el nuevo archivo limpio de {len(df_ligero)} filas...")
df_ligero.to_csv(archivo_nuevo, index=False)

print(f"Archivo purgado y reducido exitosamente en:\n{archivo_nuevo}")

### GPS User Data EDA

In [ ]:
import pandas as pd

print(f"Cargando datos GPS crudos desde {FILE_GPS_ORIGINAL}...")
df_crudo = pd.read_parquet(FILE_GPS_ORIGINAL)

# ==============================================================================
# 1. ARREGLAR RELOJ PARA EL ANÁLISIS
# ==============================================================================
col_tiempo = 'utc_timestamp' if 'utc_timestamp' in df_crudo.columns else 'date'

try:
    df_crudo[col_tiempo] = pd.to_datetime(df_crudo[col_tiempo], unit='s')
except ValueError:
    df_crudo[col_tiempo] = pd.to_datetime(df_crudo[col_tiempo])

df_crudo = df_crudo.rename(columns={col_tiempo: 'local_timestamp'})

# ==============================================================================
# 2. ANÁLISIS DE TIEMPO (MICRO-TEMBLORES)
# ==============================================================================
print("Calculando diferencias de tiempo...")
# Ordenar por usuario y tiempo para que la diferencia sea secuencial
df_crudo = df_crudo.sort_values(by=['caid', 'local_timestamp'])

# Calcular la diferencia de tiempo entre un ping y el anterior (en segundos)
df_crudo['tiempo_entre_pings_segundos'] = df_crudo.groupby('caid')['local_timestamp'].diff().dt.total_seconds()

# Limpiar el primer ping de cada usuario (que da nulo al no tener ping anterior)
df_crudo = df_crudo.dropna(subset=['tiempo_entre_pings_segundos'])

# REPORTE ESTADÍSTICO
print("\n=================================================================")
print("ESTUDIO DE FRECUENCIA DE PINGS (DATOS CRUDOS)")
print("=================================================================")
print(f"Total de pings evaluados: {len(df_crudo):,}\n")

print("▶ Distribución General del tiempo entre pings (Segundos):")
print(df_crudo['tiempo_entre_pings_segundos'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\n▶ Análisis de Micro-Temblores (Ráfagas rápidas):")
rafagas_1s = len(df_crudo[df_crudo['tiempo_entre_pings_segundos'] <= 1.0])
rafagas_5s = len(df_crudo[df_crudo['tiempo_entre_pings_segundos'] <= 5.0])
rafagas_10s = len(df_crudo[df_crudo['tiempo_entre_pings_segundos'] <= 10.0])

pct_1s = (rafagas_1s / len(df_crudo)) * 100
pct_5s = (rafagas_5s / len(df_crudo)) * 100

print(f"  - Pings casi instantáneos (<= 1 seg): {rafagas_1s:,} ({pct_1s:.2f}%)")
print(f"  - Pings en ráfaga corta (<= 5 seg): {rafagas_5s:,} ({pct_5s:.2f}%)")
print(f"  - Pings en rango normal bajo (<= 10 seg): {rafagas_10s:,}")

print("\n=================================================================")